In [ ]:
import pandas as pd

FILE_DATA = "../data/raw/data karhutla gabungan terbaru.csv"   # <- ganti ini nanti
df = pd.read_csv(FILE_DATA)
print(df.shape)
df[["comment_code", "tanggal", "komentar_bersih", "like_count", "reply_count"]].head()

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")

stop = set(stopwords.words("indonesian"))
stop |= {"yg", "gak", "ga", "nggak", "tp", "dgn", "aja", "sih", "deh", "kok",
         "udah", "udh", "nya", "jd", "utk", "krn", "dr", "sm", "bgt", "banget",
         "emang", "kan", "lah", "mah", "jg", "juga", "lg", "lagi", "gue", "gua",
         "lu", "lo", "aku", "kita", "kami", "kalo", "kalau", "biar", "tuh", "nih"}

def tokenisasi(teks):
    return [t for t in teks.split() if t not in stop and len(t) > 2 and not t.isdigit()]

df["token"] = df["komentar_bersih"].map(tokenisasi)
df = df[df["token"].str.len() >= 3].reset_index(drop=True)
print(len(df))
df[["komentar_bersih", "token"]].head(10)

In [ ]:
from collections import Counter

print(len(df), "komentar")
print(df["ada_kata_kunci"].value_counts())

hitung = Counter(t for toks in df["token"] for t in toks)
print(len(hitung), "kata unik")
hitung.most_common(50)

In [ ]:
stop |= set(stopwords.words("english"))
stop |= {"kak", "kakak", "yaa", "yaaa", "sampe", "amin", "aamiin", "wkwk", "wkwkw",
         "wkwkwk", "gk", "gtu", "gitu", "gini", "bener2", "orng", "udh", "tau"}

df["token"] = df["komentar_bersih"].map(tokenisasi)
df = df[df["token"].str.len() >= 3].reset_index(drop=True)
print(len(df), "komentar")

In [ ]:
from gensim.corpora import Dictionary

kamus = Dictionary(df["token"])
kamus.filter_extremes(no_below=5, no_above=0.5)   # buang kata terlalu langka / terlalu umum
korpus = [kamus.doc2bow(t) for t in df["token"]]

keep = [i for i, b in enumerate(korpus) if len(b) >= 3]
df = df.iloc[keep].reset_index(drop=True)
korpus = [korpus[i] for i in keep]
print(len(kamus), "kata di kamus,", len(df), "komentar")

In [ ]:
from gensim.models import LdaModel, CoherenceModel

hasil = []
for k in range(3, 9):
    m = LdaModel(korpus, num_topics=k, id2word=kamus, passes=10, random_state=42)
    c = CoherenceModel(model=m, texts=df["token"], dictionary=kamus,
                       coherence="c_v", processes=1).get_coherence()
    hasil.append({"jumlah_topik": k, "coherence": c})
    print(k, round(c, 4))

In [ ]:
metrics = pd.DataFrame(hasil)
print(metrics)

for k in (4, 5):
    m = LdaModel(korpus, num_topics=k, id2word=kamus, passes=10, random_state=42)
    print(f"\n=== K = {k} ===")
    for i in range(k):
        kata = [w for w, _ in m.show_topic(i, topn=10)]
        print(f"Topik {i}:", ", ".join(kata))

In [ ]:
import numpy as np

K = 4
model = LdaModel(korpus, num_topics=K, id2word=kamus, passes=10, random_state=42)

# topik dominan tiap komentar
dominan = []
for bow in korpus:
    dist = model.get_document_topics(bow, minimum_probability=0.0)
    tid, p = max(dist, key=lambda x: x[1])
    dominan.append((tid, p))
df["topic_id"] = [d[0] for d in dominan]
df["topic_prob"] = [d[1] for d in dominan]

for i in range(K):
    print(f"\n=== Topik {i}: {', '.join(w for w, _ in model.show_topic(i, topn=8))}")
    print("Jumlah komentar:", (df["topic_id"] == i).sum())
    contoh = df[df["topic_id"] == i].sort_values("topic_prob", ascending=False).head(3)
    for teks in contoh["komentar_asli"]:
        print("  -", teks[:150])

In [ ]:
dup = df["komentar_bersih"].duplicated(keep=False)
print("komentar dengan teks kembar:", dup.sum(), "dari", len(df))
print(df[dup]["komentar_bersih"].str[:80].value_counts().head(8))

print("\nContoh acak Topik 2:")
for teks in df[df["topic_id"] == 2].sample(8, random_state=1)["komentar_asli"]:
    print("-", teks[:150])

In [ ]:
import os
os.makedirs("../data/model_output", exist_ok=True)

NAMA_TOPIK = {
    0: "Seruan Aksi dan Kritik ke Pemerintah",
    1: "Penyebab dan Pembakaran Sengaja",
    2: "Doa dan Dukungan untuk Warga",
    3: "Laporan Kejadian dan Kabut Asap",
}
df["topic_name"] = df["topic_id"].map(NAMA_TOPIK)

# 1. topic_assignments
df[["comment_code", "topic_id", "topic_name", "topic_prob"]] \
    .rename(columns={"comment_code": "comment_id"}) \
    .to_csv("../data/model_output/topic_assignments.csv", index=False)

# 2. topic_words (p_topik dan p_korpus dipakai buat hitung lambda di dashboard)
total = np.zeros(len(kamus))
for bow in korpus:
    for wid, c in bow:
        total[wid] += c
p_korpus = total / total.sum()

baris = []
for i in range(K):
    for wid, p in model.get_topic_terms(i, topn=100):
        baris.append({"topic_id": i, "topic_name": NAMA_TOPIK[i], "word": kamus[wid],
                      "p_topik": p, "p_korpus": p_korpus[wid]})
pd.DataFrame(baris).to_csv("../data/model_output/topic_words.csv", index=False)

# 3. topic_metrics
metrics.to_csv("../data/model_output/topic_metrics.csv", index=False)

print("selesai")